# Model preparation

This notebook turns the preprocessed used-car data into train/test matrices for modelling. It deliberately fits imputers, encoders, and scaling **only on the training data** to avoid test-set leakage.

The generated artifacts are saved under a date-versioned `Data/model_ready/YYYY-MM-DD/` directory. Files inside use concise, stable names and include the fitted preprocessing pipeline, sparse feature matrices, targets, row identifiers, and feature names.

In [1]:
from pathlib import Path
import inspect

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', None)

def find_project_root(start: Path) -> Path:
    """Find the repository root when the notebook runs from any directory."""
    for candidate in (start, *start.parents):
        if (candidate / 'Data' / 'preprocessed').is_dir():
            return candidate
    raise FileNotFoundError('Could not find preprocessed data under Data/preprocessed/.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SNAPSHOT_DATE = '2026-05-25'  # YYYY-MM-DD; must match the preprocessed input
INPUT_PATH = PROJECT_ROOT / 'Data' / 'preprocessed' / f'{SNAPSHOT_DATE}_used_cars_preprocessed.csv'
OUTPUT_DIR = PROJECT_ROOT / 'Data' / 'model_ready' / SNAPSHOT_DATE
ARTIFACT_PREFIX = 'used_cars'

RANDOM_STATE = 2_300_632
TEST_SIZE = 0.20


In [2]:
cars = pd.read_csv(INPUT_PATH)

required_columns = {
    'listing_id', 'price', 'year', 'mileage_km', 'transmission',
    'fuel', 'brand', 'model', 'governorate',
}
missing_columns = required_columns.difference(cars.columns)
assert not missing_columns, f'Missing expected columns: {sorted(missing_columns)}'
assert cars['listing_id'].is_unique, 'listing_id must be unique.'
assert cars['price'].gt(0).all(), 'Price must be strictly positive.'

print(f'Loaded {cars.shape[0]:,} rows and {cars.shape[1]} columns from {INPUT_PATH.name}.')
display(cars.head())
display(cars.isna().sum().rename('missing_values').to_frame())


Loaded 8,621 rows and 9 columns from 2026-05-25_used_cars_preprocessed.csv.


,listing_id,price,year,mileage_km,transmission,fuel,brand,model,governorate
0,4672917,280000,2004,196122,Manual,Gas,Daewoo,Nubira 2,Cairo
1,6837266,700000,2011,170000,Automatic,Gas,Kia,Carens,ALSharqia
2,6867836,360000,2000,275000,Automatic,Gas,Skoda,Octavia A4,Ismailia
3,6881530,300000,2010,110000,Manual,Gas,Speranza,M11,Helwan
4,6884619,820000,2019,129000,Automatic,Gas,Subaru,Impreza,Cairo


,missing_values
listing_id,0
price,0
year,0
mileage_km,0
transmission,0
fuel,0
brand,0
model,0
governorate,0


## Define features and split the data

`price` is the prediction target. `listing_id` is retained only for traceability and is never supplied to the model. We stratify approximately by price decile when possible so both splits have comparable price ranges.

In [3]:
TARGET_COLUMN = 'price'
ID_COLUMN = 'listing_id'
NUMERIC_FEATURES = ['year', 'mileage_km']
CATEGORICAL_FEATURES = ['transmission', 'fuel', 'brand', 'model', 'governorate']
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = cars.loc[:, FEATURE_COLUMNS].copy()
y = cars[TARGET_COLUMN].copy()
listing_ids = cars[ID_COLUMN].copy()

# qcut can drop duplicate edges; fall back gracefully if stratification is not viable.
price_bins = pd.qcut(y, q=10, duplicates='drop')
stratify_by = price_bins if price_bins.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, listing_ids,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_by,
)

print(f'Training rows: {len(X_train):,}')
print(f'Test rows:     {len(X_test):,}')
print(f'Predictor columns: {FEATURE_COLUMNS}')


Training rows: 6,896
Test rows:     1,725
Predictor columns: ['year', 'mileage_km', 'transmission', 'fuel', 'brand', 'model', 'governorate']


## Fit transformations on the training split

Numeric columns are median-imputed then standardized. Categorical columns are most-frequent-imputed then one-hot encoded. `handle_unknown='ignore'` makes the pipeline safe for new brands, models, or locations encountered after training.

In [4]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# Support both recent and older scikit-learn versions.
one_hot_options = {'handle_unknown': 'ignore'}
if 'sparse_output' in inspect.signature(OneHotEncoder).parameters:
    one_hot_options['sparse_output'] = True
else:
    one_hot_options['sparse'] = True

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(**one_hot_options)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, NUMERIC_FEATURES),
        ('categorical', categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
)

# Fit only on X_train; transforming X_test afterwards prevents leakage.
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

assert X_train_ready.shape[1] == X_test_ready.shape[1] == len(feature_names)
print(f'Encoded feature count: {len(feature_names):,}')
print(f'Training matrix shape: {X_train_ready.shape}')
print(f'Test matrix shape:     {X_test_ready.shape}')


Encoded feature count: 895
Training matrix shape: (6896, 895)
Test matrix shape:     (1725, 895)


In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save sparse matrices to keep the one-hot encoded data compact.
x_train_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_x_train.npz'
x_test_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_x_test.npz'
y_train_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_y_train.csv'
y_test_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_y_test.csv'
feature_names_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_feature_names.csv'
pipeline_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_preprocessing_pipeline.joblib'

sparse.save_npz(x_train_path, X_train_ready)
sparse.save_npz(x_test_path, X_test_ready)

pd.DataFrame({'listing_id': id_train, 'price': y_train}).to_csv(
    y_train_path, index=False
)
pd.DataFrame({'listing_id': id_test, 'price': y_test}).to_csv(
    y_test_path, index=False
)
pd.DataFrame({'feature_name': feature_names}).to_csv(
    feature_names_path, index=False
)
joblib.dump(preprocessor, pipeline_path)

artifacts = pd.DataFrame({
    'artifact': [
        x_train_path.name, x_test_path.name, y_train_path.name, y_test_path.name,
        feature_names_path.name, pipeline_path.name,
    ],
    'purpose': [
        'Encoded and scaled training predictors',
        'Encoded and scaled test predictors',
        'Training target and listing identifiers',
        'Test target and listing identifiers',
        'Column order for the sparse matrices',
        'Fitted transformer for future listings',
    ],
})
print(f'Saved model-ready artifacts to: {OUTPUT_DIR}')
display(artifacts)


Saved model-ready artifacts to: /Users/yassien/Documents/Uni/MachineLearning/Project/Data/model_ready/2026-05-25


,artifact,purpose
0,used_cars_x_train.npz,Encoded and scaled training predictors
1,used_cars_x_test.npz,Encoded and scaled test predictors
2,used_cars_y_train.csv,Training target and listing identifiers
3,used_cars_y_test.csv,Test target and listing identifiers
4,used_cars_feature_names.csv,Column order for the sparse matrices
5,used_cars_preprocessing_pipeline.joblib,Fitted transformer for future listings


In [6]:
# Reload a saved artifact to verify that the hand-off to modelling is complete.
saved_X_train = sparse.load_npz(x_train_path)
saved_y_train = pd.read_csv(y_train_path)
saved_pipeline = joblib.load(pipeline_path)

assert saved_X_train.shape == X_train_ready.shape
assert len(saved_y_train) == X_train_ready.shape[0]
assert list(saved_pipeline.get_feature_names_out()) == list(feature_names)
print('Validation passed: model-ready train/test artifacts can be loaded successfully.')


Validation passed: model-ready train/test artifacts can be loaded successfully.
